In [1]:
import pandas as pd

commits = pd.read_csv("pvathana_project_summary.csv", sep=";")
stats = pd.read_csv("pvathana_project_stats.csv", sep=";")

print("Projects:", commits["project_wocid"].nunique())
print("Total commits:", len(commits))

stats.head()

Projects: 10
Total commits: 13268


,project,number of commits,number of authors,max time,min time,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,number of stars,number of forks,last commit date,ActivityPattern
0,mucollective_multidy,120,15,2025-10-08 20:48:42,2019-06-10 14:19:20,2023-04,2024-06,15,11,3,67,5,2026-09-11,declining
1,askurz_doing-bayesian-data-analysis-in-brms-an...,258,7,2025-04-15 14:27:35,2018-07-30 23:54:19,2023-02,2024-03,14,18,7,154,43,2026-01-04,irregular
2,esmvalgroup_esmvaltool,10353,295,2026-04-17 11:04:12,2017-01-26 14:15:37,NaN,NaN,0,0,0,276,153,2026-09-22,irregular
3,rtiinternational_smart,540,33,2025-02-01 06:42:54,2017-05-18 17:39:52,2019-07,2019-11,5,311,3,232,31,2024-12-02,irregular
4,jannisborn_covid_detector,140,18,2024-07-09 17:56:57,2020-03-28 15:17:49,2022-05,2023-08,16,6,3,174,81,2025-04-18,declining


In [2]:
gap_commits = []

for _, row in stats.iterrows():
    project = row["project"]

    project_commits = commits[
        commits["project_wocid"] == project
    ].copy()

    project_commits["date"] = pd.to_datetime(
        project_commits["time"], unit="s", utc=True
    )

    project_commits = project_commits.sort_values("date")

    gap_start = row["LongestGapStart"]
    gap_end = row["LongestGapEnd"]

    if pd.isna(gap_start) or pd.isna(gap_end):
        print("No gap for:", project)
        continue

    before = project_commits[
        project_commits["date"].dt.strftime("%Y-%m") < gap_start
    ].tail(10).copy()

    after = project_commits[
        project_commits["date"].dt.strftime("%Y-%m") > gap_end
    ].head(10).copy()

    before["pre/post"] = "pre"
    after["pre/post"] = "post"

    gap_commits.append(before)
    gap_commits.append(after)

gap_df = pd.concat(gap_commits, ignore_index=True)

gap_df = gap_df[
    ["pre/post", "project_wocid", "commit_sha1",
     "author", "time", "commit message"]
]

gap_df.head()

No gap for: esmvalgroup_esmvaltool


,pre/post,project_wocid,commit_sha1,author,time,commit message
0,pre,mucollective_multidy,00fc9900756b6e904d78a5e15eb048b1c8e82576,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1659475429,custom.js changes\n
1,pre,mucollective_multidy,155dc3d983ba58196a1e9695c6729eec9e609a76,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1659475520,Merge branch 'tangle' of https://github.com/MU...
2,pre,mucollective_multidy,4a56c5f7775e36ddb78c55ded97ff4888996084a,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1664244998,tests pass\n
3,pre,mucollective_multidy,eeebe7affb292f5f291060d5f82431da681dcfa9,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1664377602,"shift from future to furrr, export_json and co..."
4,pre,mucollective_multidy,4a7d4da5cc4d2e52a3e57e63e63bbd0b2f237ffc,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1664570122,code cleanup and tests for export_json and lin...


In [3]:
gap_df.to_csv(
    "pvathana_project_gap_commits.csv",
    sep=";",
    index=False
)

print("Saved:", len(gap_df), "commits")
print(gap_df.groupby(["project_wocid", "pre/post"]).size())

Saved: 171 commits
project_wocid                                                  pre/post
askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse  post        10
                                                               pre         10
ccsb-scripps_autodock-gpu                                      post        10
                                                               pre         10
ebivariation_eva-tools                                         post        10
                                                               pre         10
jannisborn_covid_detector                                      post         6
                                                               pre         10
mucollective_multidy                                           post        10
                                                               pre         10
opendatacube_datacube-alchemist                                post        10
                                                   

In [4]:
for project in gap_df["project_wocid"].unique():
    print("\nPROJECT:", project)

    project_data = gap_df[
        gap_df["project_wocid"] == project
    ]

    for period in ["pre", "post"]:
        print("\n", period.upper(), "GAP COMMITS")

        messages = project_data[
            project_data["pre/post"] == period
        ]["commit message"]

        for message in messages:
            print("-", str(message).replace("\n", " ")[:200])


PROJECT: mucollective_multidy

 PRE GAP COMMITS
- custom.js changes 
- Merge branch 'tangle' of https://github.com/MUCollective/multiverse into tangle 
- tests pass 
- shift from future to furrr, export_json and code pretty printing updates 
- code cleanup and tests for export_json and linear execution 
- parallel execution fixes + extended functionality to execute partial multiverse in parallel 
- stylise code and extract branches for mvis 
- fix to take into account arbitrary multiverse specifications by sorting on dictionary keys 
- some changes to Rmd files 
- merge 

 POST GAP COMMITS
- Fix typo 
- Merge branch 'hadley-evaluate-1.0.0' fixes for evaluate update 
- vignette updates 
- updated documentation and gh pages 
- other fixes 
- warning message for asymmetric trees #100 
- updates to the knitr multiverse notebook 
- rbuildignore changes 
- Clarified explanation 
- Issue 129 (#130)  * Issue 129  * Clarified explanation  * Clarified original

PROJECT: askurz_doing-bayesian-da

In [6]:
themes = {
    "mucollective_multidy": (
        "Feature development:Bug fixes",
        "Documentation updates:Bug fixes"
    ),
    "askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse": (
        "Documentation updates:Other",
        "Documentation updates:Feature development"
    ),
    "esmvalgroup_esmvaltool": (
        "N/A",
        "N/A"
    ),
    "rtiinternational_smart": (
        "Documentation updates:Bug fixes",
        "Bug fixes:Feature development"
    ),
    "jannisborn_covid_detector": (
        "Documentation updates:Dependency updates",
        "Documentation updates:Other"
    ),
    "overlordgolddragon_ssqueezepy": (
        "Bug fixes:Dependency updates",
        "Feature development:Documentation updates"
    ),
    "rust-bio_rust-bio": (
        "Bug fixes:Documentation updates",
        "Release:Feature development"
    ),
    "ccsb-scripps_autodock-gpu": (
        "Documentation updates:Feature development",
        "Feature development:Bug fixes"
    ),
    "ebivariation_eva-tools": (
        "Feature development:Bug fixes",
        "Bug fixes:Other"
    ),
    "opendatacube_datacube-alchemist": (
        "Automated bot contributions:Dependency updates",
        "Automated bot contributions:Bug fixes"
    )
}

stats["BeforeThemes"] = stats["project"].map(
    lambda p: themes[p][0]
)

stats["AfterThemes"] = stats["project"].map(
    lambda p: themes[p][1]
)

stats.to_csv(
    "pvathana_project_stats.csv",
    sep=";",
    index=False
)

stats[["project", "BeforeThemes", "AfterThemes"]]

,project,BeforeThemes,AfterThemes
0,mucollective_multidy,Feature development:Bug fixes,Documentation updates:Bug fixes
1,askurz_doing-bayesian-data-analysis-in-brms-an...,Documentation updates:Other,Documentation updates:Feature development
2,esmvalgroup_esmvaltool,N/A,N/A
3,rtiinternational_smart,Documentation updates:Bug fixes,Bug fixes:Feature development
4,jannisborn_covid_detector,Documentation updates:Dependency updates,Documentation updates:Other
5,overlordgolddragon_ssqueezepy,Bug fixes:Dependency updates,Feature development:Documentation updates
6,rust-bio_rust-bio,Bug fixes:Documentation updates,Release:Feature development
7,ccsb-scripps_autodock-gpu,Documentation updates:Feature development,Feature development:Bug fixes
8,ebivariation_eva-tools,Feature development:Bug fixes,Bug fixes:Other
9,opendatacube_datacube-alchemist,Automated bot contributions:Dependency updates,Automated bot contributions:Bug fixes


In [8]:
new_columns = [
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Currentstatus",
    "RecentThemes",
    "Notes"
]

for col in new_columns:
    if col not in stats.columns:
        stats[col] = ""

stats[["project"] + new_columns].head()

,project,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Currentstatus,RecentThemes,Notes
0,mucollective_multidy,,,,,,,
1,askurz_doing-bayesian-data-analysis-in-brms-an...,,,,,,,
2,esmvalgroup_esmvaltool,,,,,,,
3,rtiinternational_smart,,,,,,,
4,jannisborn_covid_detector,,,,,,,


In [9]:
commits["date"] = pd.to_datetime(
    commits["time"], unit="s", utc=True
)

cutoff = pd.Timestamp("2025-04-01", tz="UTC")
reference = pd.Timestamp("2025-10-01", tz="UTC")

for i, row in stats.iterrows():
    project = row["project"]

    recent = commits[
        (commits["project_wocid"] == project) &
        (commits["date"] >= cutoff) &
        (commits["date"] < reference)
    ]

    stats.loc[i, "Currentstatus"] = (
        "Active" if len(recent) > 0 else "Inactive"
    )

stats[["project", "Currentstatus"]]

,project,Currentstatus
0,mucollective_multidy,Active
1,askurz_doing-bayesian-data-analysis-in-brms-an...,Active
2,esmvalgroup_esmvaltool,Active
3,rtiinternational_smart,Inactive
4,jannisborn_covid_detector,Inactive
5,overlordgolddragon_ssqueezepy,Active
6,rust-bio_rust-bio,Active
7,ccsb-scripps_autodock-gpu,Inactive
8,ebivariation_eva-tools,Active
9,opendatacube_datacube-alchemist,Active


In [10]:
recent_commits = []

for project in stats["project"]:
    project_data = commits[
        commits["project_wocid"] == project
    ].copy()

    # Use commits on or before the assignment's reference date
    project_data = project_data[
        project_data["date"] < reference
    ]

    project_data = project_data.sort_values(
        "date", ascending=False
    ).head(10)

    recent_commits.append(project_data)

    print("\nPROJECT:", project)
    for msg in project_data["commit message"]:
        print("-", str(msg).replace("\n", " ")[:200])


PROJECT: mucollective_multidy
- Issue 129 (#130)  * Issue 129  * Clarified explanation  * Clarified original
- Clarified explanation 
- rbuildignore changes 
- updates to the knitr multiverse notebook 
- warning message for asymmetric trees #100 
- other fixes 
- updated documentation and gh pages 
- vignette updates 
- Merge branch 'hadley-evaluate-1.0.0' fixes for evaluate update 
- Fix typo 

PROJECT: askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse
- mutate_ 
- {4} 
- delete 
- switch bookdown for quarto 
- switch bookdown for quarto 
- switch bookdown for quarto 
- switch bookdown for quarto 
- switch bookdown for quarto 
- switch bookdown for quarto 
- switch bookdown for quarto 

PROJECT: esmvalgroup_esmvaltool
- Switch from Prospector to ruff 
- Merge eee7b1403f9554c54cfcf689641a7d9c6c2d8455 into 3795bdcaba7ed5b2ca2073b1cd2cb5461de317de 
- Merge 54fa687205fac70fa0f4f39788eac5347150ba4a into 3795bdcaba7ed5b2ca2073b1cd2cb5461de317de 
- Fixing `surface_trace_gases/di

In [11]:
for project in gap_df["project_wocid"].unique():
    data = gap_df[gap_df["project_wocid"] == project]

    before_authors = set(
        data[data["pre/post"] == "pre"]["author"]
    )

    after_authors = set(
        data[data["pre/post"] == "post"]["author"]
    )

    print("\n", project)
    print("Before:", before_authors)
    print("After:", after_authors)
    print("Returning:", before_authors & after_authors)
    print("New in sample:", after_authors - before_authors)


 mucollective_multidy
Before: {'Abhraneel Sarma <abhraneelsarma2024@u.northwestern.edu>', 'Kyle Hwang <60527858+hwangkyle@users.noreply.github.com>'}
After: {'emstruong <112967876+emstruong@users.noreply.github.com>', 'Hadley Wickham <h.wickham@gmail.com>', 'Michael S. Truong <112967876+emstruong@users.noreply.github.com>', 'Abhraneel Sarma <abhraneelsarma2024@u.northwestern.edu>', 'Abhraneel Sarma <abhraneel@Abhraneels-MacBook.local>'}
Returning: {'Abhraneel Sarma <abhraneelsarma2024@u.northwestern.edu>'}
New in sample: {'Abhraneel Sarma <abhraneel@Abhraneels-MacBook.local>', 'emstruong <112967876+emstruong@users.noreply.github.com>', 'Michael S. Truong <112967876+emstruong@users.noreply.github.com>', 'Hadley Wickham <h.wickham@gmail.com>'}

 askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse
Before: {'Solomon Kurz <a.solomon.kurz@gmail.com>'}
After: {'Solomon Kurz <solomonkurz@Solomons-MacBook-Pro.local>'}
Returning: set()
New in sample: {'Solomon Kurz <solomonkurz@Solomo

In [17]:
def get_name(author):
    name = str(author).split("<")[0].strip().lower()
    return name

for project in gap_df["project_wocid"].unique():
    data = gap_df[gap_df["project_wocid"] == project]

    before = set(
        data[data["pre/post"] == "pre"]["author"].apply(get_name)
    )

    after = set(
        data[data["pre/post"] == "post"]["author"].apply(get_name)
    )

    print("\n", project)
    print("Returning:", before & after)
    print("New in sample:", after - before)


 mucollective_multidy
Returning: {'abhraneel sarma'}
New in sample: {'michael s. truong', 'hadley wickham', 'emstruong'}

 askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse
Returning: {'solomon kurz'}
New in sample: set()

 rtiinternational_smart
Returning: {'wengerm50'}
New in sample: {'caroline kery', 'dependabot[bot]', 'astridkery'}

 jannisborn_covid_detector
Returning: {'jannis born'}
New in sample: {'lingyigt', 'jannisborn'}

 overlordgolddragon_ssqueezepy
Returning: {'john muradeli'}
New in sample: set()

 rust-bio_rust-bio
Returning: {'github-actions[bot]', 'minindu0'}
New in sample: {'david laehnemann', 'matheus', 'johannes köster', 'ragnar groot koerkamp', 'nikolai karpov'}

 ccsb-scripps_autodock-gpu
Returning: {'leonardo solis v'}
New in sample: {'aaron scheinberg'}

 ebivariation_eva-tools
Returning: set()
New in sample: {'unknown', 'jmmut', 'hritikgupta', 'hritik gupta', 'andresfsilva', 'andres silva', 'amila silva'}

 opendatacube_datacube-alchemist
Returnin

In [18]:
who_recovered = {
    "mucollective_multidy":
        "Abhraneel Sarma returned, along with contributions from Michael Truong and Hadley Wickham.",

    "askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse":
        "Solomon Kurz continued contributing after the gap.",

    "esmvalgroup_esmvaltool":
        "N/A - no inactivity gap.",

    "rtiinternational_smart":
        "wengerm50 returned, with contributions from Caroline Kery and Dependabot.",

    "jannisborn_covid_detector":
        "Jannis Born returned, with contributions from lingyigt.",

    "overlordgolddragon_ssqueezepy":
        "John Muradeli continued contributing after the gap.",

    "rust-bio_rust-bio":
        "minindu0 returned, alongside several other contributors including Johannes Koster and David Laehnemann.",

    "ccsb-scripps_autodock-gpu":
        "Leonardo Solis V returned, with contributions from Aaron Scheinberg.",

    "ebivariation_eva-tools":
        "Post-gap commits involved contributors including Hritik Gupta, Andres Silva, and Amila Silva.",

    "opendatacube_datacube-alchemist":
        "Dependabot and pre-commit.ci continued contributing, alongside Peter Jonsson and Paul Haesler."
}

stats["WhoRecovered"] = stats["project"].map(who_recovered)

stats[["project", "WhoRecovered"]]

,project,WhoRecovered
0,mucollective_multidy,"Abhraneel Sarma returned, along with contribut..."
1,askurz_doing-bayesian-data-analysis-in-brms-an...,Solomon Kurz continued contributing after the ...
2,esmvalgroup_esmvaltool,N/A - no inactivity gap.
3,rtiinternational_smart,"wengerm50 returned, with contributions from Ca..."
4,jannisborn_covid_detector,"Jannis Born returned, with contributions from ..."
5,overlordgolddragon_ssqueezepy,John Muradeli continued contributing after the...
6,rust-bio_rust-bio,"minindu0 returned, alongside several other con..."
7,ccsb-scripps_autodock-gpu,"Leonardo Solis V returned, with contributions ..."
8,ebivariation_eva-tools,Post-gap commits involved contributors includi...
9,opendatacube_datacube-alchemist,Dependabot and pre-commit.ci continued contrib...


In [19]:
recent_themes = {
    "mucollective_multidy":
        "Documentation updates:Bug fixes",

    "askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse":
        "Documentation updates:Other",

    "esmvalgroup_esmvaltool":
        "Bug fixes:Other",

    "rtiinternational_smart":
        "Other:Bug fixes",

    "jannisborn_covid_detector":
        "Documentation updates:Other",

    "overlordgolddragon_ssqueezepy":
        "Feature development:Documentation updates",

    "rust-bio_rust-bio":
        "Feature development:Release",

    "ccsb-scripps_autodock-gpu":
        "Bug fixes:Feature development",

    "ebivariation_eva-tools":
        "Other:Feature development",

    "opendatacube_datacube-alchemist":
        "Automated bot contributions:Dependency updates"
}

stats["RecentThemes"] = stats["project"].map(recent_themes)

stats[["project", "RecentThemes"]]

,project,RecentThemes
0,mucollective_multidy,Documentation updates:Bug fixes
1,askurz_doing-bayesian-data-analysis-in-brms-an...,Documentation updates:Other
2,esmvalgroup_esmvaltool,Bug fixes:Other
3,rtiinternational_smart,Other:Bug fixes
4,jannisborn_covid_detector,Documentation updates:Other
5,overlordgolddragon_ssqueezepy,Feature development:Documentation updates
6,rust-bio_rust-bio,Feature development:Release
7,ccsb-scripps_autodock-gpu,Bug fixes:Feature development
8,ebivariation_eva-tools,Other:Feature development
9,opendatacube_datacube-alchemist,Automated bot contributions:Dependency updates


In [20]:
gap_findings = {
    "mucollective_multidy": (
        "Unknown; development stopped after code fixes and parallel execution updates.",
        "Yes",
        "Development resumed with documentation updates, compatibility fixes, and maintenance."
    ),

    "askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse": (
        "Unknown; activity stopped after bookdown and formatting updates.",
        "Yes",
        "Solomon Kurz resumed work, including switching the project from bookdown to Quarto."
    ),

    "esmvalgroup_esmvaltool": (
        "N/A active Project",
        "N/A",
        "N/A - no inactivity gap."
    ),

    "rtiinternational_smart": (
        "Unknown; the gap followed documentation, Docker, and setup updates.",
        "Yes",
        "Development resumed with frontend changes, build fixes, and dependency updates."
    ),

    "jannisborn_covid_detector": (
        "Unknown; the gap followed documentation and dataset maintenance.",
        "Limited",
        "A small number of commits resumed, mainly documentation and repository maintenance."
    ),

    "overlordgolddragon_ssqueezepy": (
        "Unknown; activity paused after bug fixes and dependency maintenance.",
        "Limited",
        "The original maintainer returned for a few updates involving functionality and documentation."
    ),

    "rust-bio_rust-bio": (
        "Unknown; the gap followed fixes, documentation, and a release.",
        "Yes",
        "Development resumed with new features, bug fixes, and releases."
    ),

    "ccsb-scripps_autodock-gpu": (
        "Unknown; the gap followed documentation updates and GPU-related improvements.",
        "Yes",
        "Development resumed with code porting, GPU development, and licensing changes."
    ),

    "ebivariation_eva-tools": (
        "Unknown; the gap followed refactoring, bug fixes, and version updates.",
        "Yes",
        "Development resumed with bug fixes, code cleanup, and new functionality."
    ),

    "opendatacube_datacube-alchemist": (
        "Unknown; the gap followed automated dependency and CI updates.",
        "Yes",
        "Activity resumed through automated updates and human maintenance of CI and dependencies."
    )
}

stats["HypothesizedGapReason"] = stats["project"].map(
    lambda p: gap_findings[p][0]
)

stats["HasRecovered"] = stats["project"].map(
    lambda p: gap_findings[p][1]
)

stats["WhyRecovered"] = stats["project"].map(
    lambda p: gap_findings[p][2]
)

stats[
    ["project", "HypothesizedGapReason", "HasRecovered", "WhyRecovered"]
]

,project,HypothesizedGapReason,HasRecovered,WhyRecovered
0,mucollective_multidy,Unknown; development stopped after code fixes ...,Yes,Development resumed with documentation updates...
1,askurz_doing-bayesian-data-analysis-in-brms-an...,Unknown; activity stopped after bookdown and f...,Yes,"Solomon Kurz resumed work, including switching..."
2,esmvalgroup_esmvaltool,N/A active Project,N/A,N/A - no inactivity gap.
3,rtiinternational_smart,"Unknown; the gap followed documentation, Docke...",Yes,"Development resumed with frontend changes, bui..."
4,jannisborn_covid_detector,Unknown; the gap followed documentation and da...,Limited,"A small number of commits resumed, mainly docu..."
5,overlordgolddragon_ssqueezepy,Unknown; activity paused after bug fixes and d...,Limited,The original maintainer returned for a few upd...
6,rust-bio_rust-bio,"Unknown; the gap followed fixes, documentation...",Yes,"Development resumed with new features, bug fix..."
7,ccsb-scripps_autodock-gpu,Unknown; the gap followed documentation update...,Yes,"Development resumed with code porting, GPU dev..."
8,ebivariation_eva-tools,"Unknown; the gap followed refactoring, bug fix...",Yes,"Development resumed with bug fixes, code clean..."
9,opendatacube_datacube-alchemist,Unknown; the gap followed automated dependency...,Yes,Activity resumed through automated updates and...


In [21]:
notes = {
    "mucollective_multidy":
        "The project had a 15-month inactivity gap from April 2023 to June 2024. The exact reason is unclear, but commits afterward show documentation updates and compatibility fixes. Abhraneel Sarma returned, along with other contributors, suggesting that maintenance resumed.",

    "askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse":
        "The project had a 14-month gap from February 2023 to March 2024. The reason for the pause is unclear, but Solomon Kurz returned and migrated the project from bookdown to Quarto. This suggests the project resumed with a focus on updating its publishing tools.",

    "esmvalgroup_esmvaltool":
        "The project had no months without commits during the period analyzed. Its activity remained continuous, although the number of commits varied over time. This suggests ongoing development rather than abandonment and recovery.",

    "rtiinternational_smart":
        "The project had a five-month inactivity gap from July to November 2019. The reason for the gap is unknown, but development resumed with frontend improvements, build fixes, and dependency updates. Returning and additional contributors participated in the resumed work.",

    "jannisborn_covid_detector":
        "The project had a 16-month gap from May 2022 to August 2023. Only six commits followed the gap in the dataset, mostly involving documentation and maintenance. Jannis Born returned, but the limited activity suggests only a partial recovery.",

    "overlordgolddragon_ssqueezepy":
        "The project had an eight-month gap from December 2024 to July 2025. John Muradeli returned afterward and made a small number of updates to functionality, dependencies, and documentation. Because only five commits followed the gap, the recovery appears limited.",

    "rust-bio_rust-bio":
        "The project had a three-month inactivity gap from February to April 2023. Development resumed with new features, fixes, and releases involving both returning and additional contributors. The continued activity suggests that the gap was temporary.",

    "ccsb-scripps_autodock-gpu":
        "The project had an eight-month gap from July 2019 to February 2020. Development resumed with GPU-related code changes, porting work, and licensing updates. Leonardo Solis V returned alongside another contributor, suggesting a renewed development effort.",

    "ebivariation_eva-tools":
        "The project had a six-month gap from April to September 2018. Afterward, contributors made bug fixes, cleanup changes, and functional improvements. The project later continued receiving commits, suggesting that development recovered.",

    "opendatacube_datacube-alchemist":
        "The project had a five-month gap from May to September 2024. Activity resumed through automated dependency updates and human maintenance work. The continued presence of bots and human contributors suggests that development resumed, although much of the activity involved maintenance."
}

stats["Notes"] = stats["project"].map(notes)

stats[["project", "Notes"]]

,project,Notes
0,mucollective_multidy,The project had a 15-month inactivity gap from...
1,askurz_doing-bayesian-data-analysis-in-brms-an...,The project had a 14-month gap from February 2...
2,esmvalgroup_esmvaltool,The project had no months without commits duri...
3,rtiinternational_smart,The project had a five-month inactivity gap fr...
4,jannisborn_covid_detector,The project had a 16-month gap from May 2022 t...
5,overlordgolddragon_ssqueezepy,The project had an eight-month gap from Decemb...
6,rust-bio_rust-bio,The project had a three-month inactivity gap f...
7,ccsb-scripps_autodock-gpu,The project had an eight-month gap from July 2...
8,ebivariation_eva-tools,The project had a six-month gap from April to ...
9,opendatacube_datacube-alchemist,The project had a five-month gap from May to S...


In [22]:
columns = [
    "project",
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Currentstatus",
    "RecentThemes",
    "Notes"
]

print(stats[columns].isna().sum())

project                  0
HypothesizedGapReason    0
HasRecovered             0
WhyRecovered             0
WhoRecovered             0
Currentstatus            0
RecentThemes             0
Notes                    0
dtype: int64


In [27]:
evidence = {
    "askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse":
        "The repository's README confirms that the project uses Quarto. "
        "This supports the commit-message evidence of a publishing-system migration, "
        "but does not establish why the gap occurred.",

    "rust-bio_rust-bio":
        "The project changelog documents releases 1.2.0 and 1.3.0 "
        "in June 2023, supporting the conclusion that development resumed.",

    "ccsb-scripps_autodock-gpu":
        "The repository documents CUDA development and collaboration. "
        "Post-gap commits show GPU-related development, but the cause of the gap is unknown.",

    "overlordgolddragon_ssqueezepy":
        "GitHub issues show that users continued requesting functionality "
        "during the inactive period. This does not establish why commits stopped.",

    "opendatacube_datacube-alchemist":
        "The repository documentation describes a dataset-transformation tool. "
        "Commit messages show dependency and CI maintenance after the gap."
}

stats["Evidence"] = stats["project"].map(evidence).fillna(
    "Commit messages provide evidence of work before and after the gap, "
    "but no independent explanation for the inactivity was established."
)

print("Evidence added:", len(stats))

Evidence added: 10


In [28]:
for _, row in stats.iterrows():
    project = row["project"]

    reflection = f"""# {project} Reflection

## Activity Pattern
{row['ActivityPattern']}

## Longest Inactivity Gap
Start: {row['LongestGapStart']}
End: {row['LongestGapEnd']}
Length: {row['LongestGapLength']} months

## Interpretation
{row['Notes']}

Hypothesized reason: {row['HypothesizedGapReason']}

## Supporting Evidence
{row['Evidence']}

## Recovery
Recovered: {row['HasRecovered']}
Why: {row['WhyRecovered']}
Who: {row['WhoRecovered']}

## Current Status
{row['Currentstatus']}

## Recent Commit Themes
{row['RecentThemes']}
"""

    with open(f"pvathana_reflection_{project}.md", "w", encoding="utf-8") as f:
        f.write(reflection)

print("Updated all 10 reflection files.")

Updated all 10 reflection files.


In [29]:
stats.drop(columns=["Evidence"]).to_csv(
    "pvathana_project_stats.csv",
    sep=";",
    index=False
)

print("Saved!")

Saved!
